# VisionAgent

> Notebook generated from [`examples/multimodal/01_vision_agent.py`](../../examples/multimodal/01_vision_agent.py).

| Field | Value |
|------|-------|
| **Dataset** | arXiv cs.CV (local CSV, synthetic figures) |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** Analysis without OCR, with OCR, and MediaValidator fallback against a non-image blob.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
VisionAgent — Image analysis with injectable vision LLM
=============================================================
Component: SPEC-MM-AGT-001 / prismal.agents.multimodal.vision_agent

Dataset: arXiv Computer Vision (hypothetical figures from the abstracts)
  • Subset of cs.CV papers from
    `Langgraph_tutorials/data/arxiv/arxiv_papers.csv`.
  • For each paper we build a "synthetic figure" (minimal PNG) and
    associate an expected caption derived from the title/abstract — so
    we can test the agent without a real VLM.
  • Why arXiv: covers the full spectrum of computer vision
    (detection, segmentation, video, OCR), so it serves to
    demonstrate `with_ocr=True/False`, validation fallbacks and the
    `VisionResult` contract.

Component description:
  The agent receives `image: bytes | Path` and an optional `prompt`,
  runs it through `MediaValidator` (magic bytes + byte limit)
  and delegates the description to the `vision_fn` callable. If `with_ocr=True`,
  it performs a second OCR pass. It always returns a
  `VisionResult(description, objects, ocr_text, model_used,
  used_fallback)`; it never throws if `degrade_gracefully=True`.

  Layered defaults — `_make_default_vision_fn` wires up the real call to
  `get_vision_llm` with `SecurePromptBuilder`. Injecting a callable
  completely bypasses the network (this example).

Usage:
    uv run python examples/multimodal/01_vision_agent.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Locate the dataset

This notebook reads data from `data/` in the repo root. The cell below searches for that folder by walking up from the current cwd.

In [ ]:
# Resolve the repo's data/ folder by walking up until it is found.
# Replaces DATA_ROOT
# that the original script had when it lived inside prismal/examples/.
import pathlib


def _find_data_root() -> pathlib.Path:
    cur = pathlib.Path.cwd().resolve()
    for _ in range(8):
        if (cur / "data").is_dir():
            return cur / "data"
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        "data/ folder not found. Run the notebook from the "
        "prisma-notebooks/ repo where data/ is at the root."
    )


DATA_ROOT = _find_data_root()
print(f"DATA_ROOT = {DATA_ROOT}")

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import csv
import struct
import zlib
from dataclasses import dataclass
from pathlib import Path

from prismal.agents.multimodal import VisionAgent, VisionResult

## Dataset: arXiv cs.CV papers (same CSV used in multivector RAG)

In [ ]:
ARXIV_CSV = (
    DATA_ROOT
    / "arxiv"
    / "arxiv_papers.csv"
)


@dataclass(frozen=True)
class PaperFigure:
    """A synthetic figure associated with an arXiv paper."""

    arxiv_id: str
    title: str
    expected_caption: str   # What an "ideal" VLM should answer.
    expected_objects: tuple[str, ...]
    image_bytes: bytes


def _png_solid(width: int, height: int, rgb: tuple[int, int, int]) -> bytes:
    """Generate a valid PNG (1 solid color) without Pillow. Demo only."""
    sig = b"\x89PNG\r\n\x1a\n"

    def _chunk(tag: bytes, data: bytes) -> bytes:
        crc = zlib.crc32(tag + data)
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", crc)

    ihdr = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    raw = b""
    row = b"\x00" + bytes(rgb) * width
    for _ in range(height):
        raw += row
    idat = zlib.compress(raw, 9)
    return sig + _chunk(b"IHDR", ihdr) + _chunk(b"IDAT", idat) + _chunk(b"IEND", b"")


def _load_dataset(limit: int = 5) -> list[PaperFigure]:
    """Load cs.CV papers and build figures + expected captions."""
    if not ARXIV_CSV.exists():
        # Embedded fallback in case the dataset is not accessible.
        return _embedded_dataset()

    rows: list[PaperFigure] = []
    with ARXIV_CSV.open(encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        for raw in reader:
            if not raw.get("category", "").startswith("cs.CV"):
                continue
            title = (raw.get("title") or "").strip()
            abstract = (raw.get("abstract") or "").strip()
            # Deterministic color per arXiv id (rotates blue/green/red).
            seed = abs(hash(raw["arxiv_id"])) % 3
            rgb = [(45, 110, 220), (60, 170, 90), (200, 80, 70)][seed]
            rows.append(
                PaperFigure(
                    arxiv_id=raw["arxiv_id"],
                    title=title,
                    expected_caption=f"Figure from {title!r}. Topic: {abstract[:140]}…",
                    expected_objects=("paper", "diagram", "figure"),
                    image_bytes=_png_solid(64, 64, rgb),
                )
            )
            if len(rows) >= limit:
                break
    return rows or _embedded_dataset()


def _embedded_dataset() -> list[PaperFigure]:
    """Plan B: 3 embedded figures if the CSV is not available."""
    samples = [
        ("2604.02330", "ActionParty: Multi-Subject World Models", "video, agent"),
        ("2604.02185", "Vision Transformers Survey", "transformer, attention"),
        ("2604.02077", "Diffusion Models for Segmentation", "diffusion, mask"),
    ]
    return [
        PaperFigure(
            arxiv_id=aid,
            title=title,
            expected_caption=f"Synthetic figure for paper {title!r} ({objs}).",
            expected_objects=tuple(objs.split(", ")),
            image_bytes=_png_solid(64, 64, (45, 110, 220)),
        )
        for aid, title, objs in samples
    ]

## Fake vision_fn — returns the expected caption without calling a VLM

In [ ]:
def make_fake_vision_fn(corpus: list[PaperFigure]):
    """Vision-fn that looks at the first bytes to identify the paper."""
    by_signature = {fig.image_bytes[:16]: fig for fig in corpus}

    async def _vision(image, prompt: str) -> str:
        blob = image if isinstance(image, bytes) else Path(image).read_bytes()
        fig = by_signature.get(blob[:16])
        if fig is None:
            return f"[mock-vlm] cannot identify image · prompt={prompt!r}"
        return fig.expected_caption

    return _vision


async def make_fake_ocr_fn(image, prompt=None):  # noqa: ARG001 - shape compat
    """OCR mock: the text is always 'arXiv preprint · 2024' as a stamp."""
    return "arXiv preprint · 2024"

## Main demo

In [ ]:
async def main() -> None:
    print("=" * 70)
    print("VisionAgent · analysis of synthetic arXiv figures")
    print("=" * 70)

    corpus = _load_dataset(limit=4)
    print(f"\nLoaded {len(corpus)} cs.CV papers\n")

    agent = VisionAgent(
        vision_fn=make_fake_vision_fn(corpus),
        ocr_fn=make_fake_ocr_fn,
        degrade_gracefully=True,
    )

    # 1. Analysis without OCR
    print("─" * 70)
    print("1) Standard analysis (with_ocr=False)")
    print("─" * 70)
    for fig in corpus:
        result: VisionResult = await agent.analyze(fig.image_bytes, with_ocr=False)
        print(f"\n  {fig.arxiv_id} — {fig.title[:60]}")
        print(f"    description: {result.description[:80]}…")
        print(f"    used_fallback: {result.used_fallback}")
        print(f"    model_used: {result.model_used}")

    # 2. Analysis with OCR
    print("\n" + "─" * 70)
    print("2) Analysis with OCR (with_ocr=True)")
    print("─" * 70)
    result = await agent.analyze(
        corpus[0].image_bytes,
        prompt="What topic does this paper cover?",
        with_ocr=True,
    )
    print(f"\n  ocr_text: {result.ocr_text!r}")
    print(f"  description: {result.description[:80]}…")

    # 3. Validation rejects junk (incorrect magic bytes)
    print("\n" + "─" * 70)
    print("3) MediaValidator rejects a non-image blob")
    print("─" * 70)
    bogus = b"this is not an image"
    result = await agent.analyze(bogus)
    assert result.used_fallback, "The agent must degrade to fallback"
    print(f"\n  used_fallback: {result.used_fallback}  ← validation blocked the VLM call")
    print(f"  description: {result.description!r}  (empty in fallback)")

    print("\n" + "=" * 70)
    print("OK — VisionAgent works with injected vision_fn (no network)")
    print("=" * 70)


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()